# Clipt Detection Models — v3 OCR Notebook
# ALI REPLACEMENT LAYER
#
# These models replace Ali's jersey OCR ensemble entirely.
# Uses YOLOv8m (8x more parameters than v2) at imgsz=832.
# Primary model trained on 13,815 images.
# Ensemble trained on 20,747 merged images.
#
# When ALL v3 models are loaded the detection pipeline is:
# 1. player_isolator_v3 — finds exact player bbox
# 2. jersey_color_classifier_v3 — confirms correct player
# 3. number_region_detector_v3 — finds number location
# 4. jersey_ocr_v3_ensemble — reads the number (primary)
# 5. Sport-specific model confirms (basketball/football/lax)
# 6. Specialist models catch edge cases (blur/dark/wide)
# 7. temporal_consensus requires 3+ frame agreement
# 8. cross_layer_boost if Ali also agrees
#
# INSTRUCTIONS:
# 1. Runtime → Change runtime type → A100 GPU
# 2. Add ROBOFLOW_API_KEY to Colab Secrets
# 3. Run chunks IN ORDER — 1, 2, 3, 4
# 4. Download after EACH chunk before moving on
# 5. Total time: ~5-6 hours on A100
#
# CHUNK STRUCTURE:
# Chunk 1 — Multi-sport OCR (~90 min) → DOWNLOAD
# Chunk 2 — Sport-specific OCR (~75 min) → DOWNLOAD
# Chunk 3 — Player isolation (~60 min) → DOWNLOAD
# Chunk 4 — Specialists (~60 min) → DOWNLOAD
#
# Training Tiers:
#   Large  (2000+): epochs=75,  degrees=15, scale=0.7
#   Medium (500-1999): epochs=100, degrees=20, scale=0.7
#   Small  (300-499):  epochs=150, degrees=25, scale=0.8
#
# Model Filename Cross-Reference (MUST match roboflow_detector.py):
# ```
# CHUNK 1 — Multi-sport Jersey OCR
#   jersey_ocr_v3_primary.pt      (13,815 images)
#   jersey_ocr_v3_secondary.pt    (6,932 images)
#   jersey_ocr_v3_ensemble.pt     (merged ~20,747 images)
#
# CHUNK 2 — Sport-specific OCR
#   basketball_ocr_v3.pt          (826 images)
#   football_ocr_v3.pt            (2,918 + supplement)
#   lacrosse_ocr_v3.pt            (2,100 images)
#
# CHUNK 3 — Player Isolation + Color
#   player_isolator_v3.pt         (5,174 images)
#   jersey_color_classifier_v3.pt (1,232 images)
#   number_region_detector_v3.pt  (556 images)
#
# CHUNK 4 — Augmentation Specialists
#   motion_blur_specialist_v3.pt
#   wide_angle_specialist_v3.pt   (imgsz=1280)
#   dark_jersey_specialist_v3.pt
#   partial_visibility_specialist_v3.pt
# ```

## Setup — Run this first (every session)

In [ ]:
import os

# ── GPU verification ──────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU"
device_name = torch.cuda.get_device_name(0)
print(f"GPU: {device_name}")
assert "A100" in device_name or "T4" in device_name, \
    "Switch to A100 for faster training"

# ── Read API key from Colab Secrets (key icon in sidebar) ─────
from google.colab import userdata
api_key = userdata.get('ROBOFLOW_API_KEY')

!pip install roboflow ultralytics -q
from roboflow import Roboflow
from ultralytics import YOLO

rf = Roboflow(api_key=api_key)

# ── v3 training constants ─────────────────────────────────────
V3_BASE = "yolov8m.pt"   # 25.9M params — medium backbone for OCR accuracy
V3_IMGSZ = 832            # Higher res for small text OCR
V3_BATCH = 8              # Larger model = smaller batch
V3_DEVICE = 0

# ── Dataset download helper ───────────────────────────────────
def safe_download(workspace, project_slug, version, fallback_version=1, fmt="yolov8"):
    """Download dataset with error handling + auto-retry on fallback version."""
    try:
        proj = rf.workspace(workspace).project(project_slug)
        ds = proj.version(version).download(fmt)
        print(f"  Downloaded: {ds.location}")
        return ds
    except Exception as e:
        print(f"  Version {version} failed: {e}")
        if version != fallback_version:
            print(f"  Retrying with version {fallback_version}...")
            try:
                ds = proj.version(fallback_version).download(fmt)
                print(f"  Downloaded (fallback): {ds.location}")
                return ds
            except Exception as e2:
                print(f"  Fallback also failed: {e2}")
        print(f"  SKIPPED — download failed completely")
        return None

# ── Dataset merge helper ──────────────────────────────────────
import shutil, yaml

def merge_datasets(dataset_a_path, dataset_b_path, merged_name):
    """Merge two YOLO datasets by copying images+labels with prefixes."""
    merged_dir = f"/content/{merged_name}"
    for split in ["train", "valid", "test"]:
        for subdir in ["images", "labels"]:
            os.makedirs(f"{merged_dir}/{split}/{subdir}", exist_ok=True)
        for ds_prefix, ds_path in [("a", dataset_a_path), ("b", dataset_b_path)]:
            for subdir in ["images", "labels"]:
                src = f"{ds_path}/{split}/{subdir}"
                dst = f"{merged_dir}/{split}/{subdir}"
                if os.path.exists(src):
                    for f in os.listdir(src):
                        shutil.copy(f"{src}/{f}", f"{dst}/{ds_prefix}_{f}")
    with open(f"{dataset_a_path}/data.yaml") as f:
        yaml_a = yaml.safe_load(f)
    merged_yaml = {
        'path': merged_dir,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'names': yaml_a['names'],
        'nc': yaml_a['nc'],
    }
    with open(f"{merged_dir}/data.yaml", 'w') as f:
        yaml.dump(merged_yaml, f)
    train_count = len(os.listdir(f"{merged_dir}/train/images"))
    print(f"Merged dataset: {train_count} training images")
    return merged_dir

# ── Download-if-pass helper ───────────────────────────────────
from google.colab import files as colab_files

def download_if_pass(model_name, run_name, min_map50=0.5):
    """Validate mAP50 and download if passing."""
    path = f"runs/detect/{run_name}/weights/best.pt"
    if not os.path.exists(path):
        print(f"MISSING: {model_name}")
        return False
    model = YOLO(path)
    metrics = model.val()
    map50 = metrics.box.map50
    if map50 >= min_map50:
        size_mb = os.path.getsize(path) / 1024 / 1024
        shutil.copy(path, model_name)
        colab_files.download(model_name)
        print(f"DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
        return True
    else:
        print(f"FAILED: {model_name} mAP50={map50:.3f} — below {min_map50} threshold")
        return False

print(f"Setup complete — GPU: {device_name}, base: {V3_BASE}, imgsz: {V3_IMGSZ}")

# ================================================================
# CHUNK 1 — Multi-sport Jersey OCR (~90 min on A100)
# ================================================================
# Three models: primary, secondary, and ensemble (merged)
# These are the core OCR models that replace Ali's single detector.
# After this chunk finishes, download all 3 files before continuing.

In [ ]:
# ── Cell 1A — Download multi-sport OCR datasets ───────────────

print("Downloading multi-sport OCR datasets...")

# Primary: largest digit-level jersey number dataset available
# 13,815 images, 1 class (digit), MIT license
print("\n1/2 Primary — digit detection (13,815 images):")
dataset_primary = safe_download(
    "footballplayertracking", "jerseynumberdetectordigitdetector", 1
)

# Secondary: multi-class jersey number detection
# 6,932 images, 12 classes (digits 0-9 + cero + n), CC BY 4.0
print("\n2/2 Secondary — jersey number detection (6,932 images):")
dataset_secondary = safe_download(
    "volleyai-actions", "jersey-number-detection-s01j4", 2
)

ready = sum(1 for d in [dataset_primary, dataset_secondary] if d is not None)
print(f"\n{'OK' if ready == 2 else 'WARNING'} {ready}/2 Chunk 1 datasets ready")

In [ ]:
# ── Cell 1B — Train jersey_ocr_v3_primary ─────────────────────
# 13,815 images — LARGE tier
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: jersey_ocr_v3_primary (13,815 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=75,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="jersey_ocr_v3_primary",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        degrees=15,
        translate=0.15,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )
    metrics = YOLO("runs/detect/jersey_ocr_v3_primary/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"jersey_ocr_v3_primary mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: jersey_ocr_v3_primary — dataset not available")

In [ ]:
# ── Cell 1C — Train jersey_ocr_v3_secondary ──────────────────
# 6,932 images — LARGE tier
if dataset_secondary is not None:
    print("=" * 60)
    print("TRAINING: jersey_ocr_v3_secondary (6,932 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_secondary.location}/data.yaml",
        epochs=75,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="jersey_ocr_v3_secondary",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        degrees=15,
        translate=0.15,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )
    metrics = YOLO("runs/detect/jersey_ocr_v3_secondary/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"jersey_ocr_v3_secondary mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: jersey_ocr_v3_secondary — dataset not available")

In [ ]:
# ── Cell 1D — Merge datasets + Train jersey_ocr_v3_ensemble ──
# Merged primary + secondary (~20,747 images) — LARGE tier
if dataset_primary is not None and dataset_secondary is not None:
    print("=" * 60)
    print("MERGING datasets for ensemble model...")
    print("=" * 60)

    merged_dir = merge_datasets(
        dataset_primary.location,
        dataset_secondary.location,
        "merged_ocr_ensemble",
    )

    print("\n" + "=" * 60)
    print("TRAINING: jersey_ocr_v3_ensemble (merged ~20,747 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{merged_dir}/data.yaml",
        epochs=75,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="jersey_ocr_v3_ensemble",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        degrees=15,
        translate=0.15,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )
    metrics = YOLO("runs/detect/jersey_ocr_v3_ensemble/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"jersey_ocr_v3_ensemble mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: jersey_ocr_v3_ensemble — need both primary + secondary datasets")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CHUNK 1 DOWNLOAD — Run this before Chunk 2
# ════════════════════════════════════════════════════════════════

download_if_pass("jersey_ocr_v3_primary.pt", "jersey_ocr_v3_primary")
download_if_pass("jersey_ocr_v3_secondary.pt", "jersey_ocr_v3_secondary")
download_if_pass("jersey_ocr_v3_ensemble.pt", "jersey_ocr_v3_ensemble")

print("\nCHUNK 1 COMPLETE — save files before Chunk 2")

# ================================================================
# CHUNK 2 — Sport-specific OCR (~75 min on A100)
# ================================================================
# Three models tuned for basketball, football, lacrosse digit patterns.
# Basketball digits tend to be larger; football has more motion blur;
# lacrosse uses general number detection (no lacrosse OCR dataset exists).
# Download all files after this chunk before continuing.

In [ ]:
# ── Cell 2A — Download sport-specific OCR datasets ────────────

print("Downloading sport-specific OCR datasets...")

# Basketball: 10 digit classes (0-9), 826 images
# NOTE: basketball-jersey-numbers-ocr (3,615 images) uses multimodal
# SmolVLM2 format, NOT standard YOLO bounding boxes — cannot use it.
print("\n1/4 Basketball digits (826 images, 10 digit classes):")
dataset_bball_ocr = safe_download(
    "dark-blue-jt0mg", "jerseynumbers", 5
)

# Football: jersey tracker, 2,918 images
print("\n2/4 Football jersey tracker (2,918 images):")
dataset_fb_ocr = safe_download(
    "football-tracking", "football-jersey-tracker", 1
)

# Football supplement: primary digit dataset for dark/navy jersey coverage
print("\n3/4 Football supplement — digit detection (13,815 images):")
if 'dataset_primary' not in dir() or dataset_primary is None:
    dataset_primary = safe_download(
        "footballplayertracking", "jerseynumberdetectordigitdetector", 1
    )
else:
    print(f"  Already loaded: {dataset_primary.location}")

# Lacrosse: no dedicated lacrosse OCR dataset exists.
# Using general number detection (10 classes: 0-9, ~2,100 images)
print("\n4/4 Lacrosse/general number detection (2,100 images):")
dataset_lax_ocr = safe_download(
    "smart-scoreboard", "number-rr9yl-nd3hg", 1
)

ready = sum(1 for d in [dataset_bball_ocr, dataset_fb_ocr, dataset_lax_ocr] if d is not None)
print(f"\n{'OK' if ready == 3 else 'WARNING'} {ready}/3 Chunk 2 datasets ready")

In [ ]:
# ── Cell 2B — Train basketball_ocr_v3 ─────────────────────────
# 826 images — MEDIUM tier
if dataset_bball_ocr is not None:
    print("=" * 60)
    print("TRAINING: basketball_ocr_v3 (826 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_bball_ocr.location}/data.yaml",
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="basketball_ocr_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    metrics = YOLO("runs/detect/basketball_ocr_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"basketball_ocr_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: basketball_ocr_v3 — dataset not available")

In [ ]:
# ── Cell 2C — Train football_ocr_v3 ──────────────────────────
# Merged football-tracker (2,918) + primary digit supplement — MEDIUM tier
# Includes dark/navy jersey images from primary dataset
if dataset_fb_ocr is not None:
    print("=" * 60)
    print("TRAINING: football_ocr_v3 (YOLOv8m)")
    print("=" * 60)

    # Merge with primary dataset for dark jersey coverage
    fb_data_path = f"{dataset_fb_ocr.location}/data.yaml"
    if dataset_primary is not None:
        try:
            fb_merged = merge_datasets(
                dataset_fb_ocr.location,
                dataset_primary.location,
                "merged_football_ocr",
            )
            fb_data_path = f"{fb_merged}/data.yaml"
            print("Using merged football + primary dataset")
        except Exception as e:
            print(f"Merge failed, using football-only: {e}")
    else:
        print("Primary dataset not available, using football-only")

    model = YOLO(V3_BASE)
    model.train(
        data=fb_data_path,
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="football_ocr_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    metrics = YOLO("runs/detect/football_ocr_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"football_ocr_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: football_ocr_v3 — dataset not available")

In [ ]:
# ── Cell 2D — Train lacrosse_ocr_v3 ──────────────────────────
# 2,100 images — SMALL tier (user override for max augmentation)
# NOTE: No dedicated lacrosse OCR dataset exists on Roboflow.
# Using general scoreboard number detection as proxy.
if dataset_lax_ocr is not None:
    print("=" * 60)
    print("TRAINING: lacrosse_ocr_v3 (2,100 images, YOLOv8m, SMALL tier)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_lax_ocr.location}/data.yaml",
        epochs=150,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="lacrosse_ocr_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.6,
        degrees=25,
        translate=0.2,
        scale=0.8,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.2,
        erasing=0.4,
    )
    metrics = YOLO("runs/detect/lacrosse_ocr_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"lacrosse_ocr_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: lacrosse_ocr_v3 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CHUNK 2 DOWNLOAD — Run before Chunk 3
# ════════════════════════════════════════════════════════════════

download_if_pass("basketball_ocr_v3.pt", "basketball_ocr_v3")
download_if_pass("football_ocr_v3.pt", "football_ocr_v3")
download_if_pass("lacrosse_ocr_v3.pt", "lacrosse_ocr_v3")

print("\nCHUNK 2 COMPLETE — save files before Chunk 3")

# ================================================================
# CHUNK 3 — Player Isolation + Color (~60 min on A100)
# ================================================================
# Three models: player bounding boxes, jersey color classification,
# and number region detection (narrowing crop to just the number).
# These feed into the OCR pipeline: isolate player -> find number
# region -> run OCR on cropped region.
# Download all files after this chunk before continuing.

In [ ]:
# ── Cell 3A — Download player isolation datasets ──────────────

print("Downloading player isolation + color datasets...")

# Player isolation: 5,174 images, player bounding boxes with team labels
# Largest player detection dataset with tight accurate bounding boxes
print("\n1/3 Player isolator (5,174 images):")
dataset_player = safe_download(
    "ai-in-sports", "football-player-identification", 1
)

# Jersey color: team color classification from player crops
# 1,232 images with team-color annotated bounding boxes
print("\n2/3 Jersey color classifier (1,232 images):")
dataset_color = safe_download(
    "augmented-startups", "football-player-detection-kucab", 1
)

# Number region: 556 images, 110 classes (numbers 0-99 + variants)
# Detects specifically WHERE the number is on a jersey (not full player)
print("\n3/3 Number region detector (556 images, 110 classes):")
dataset_numregion = safe_download(
    "yakovk", "jersey-numbers-i1wn5", 1
)

ready = sum(1 for d in [dataset_player, dataset_color, dataset_numregion] if d is not None)
print(f"\n{'OK' if ready == 3 else 'WARNING'} {ready}/3 Chunk 3 datasets ready")

In [ ]:
# ── Cell 3B — Train player_isolator_v3 ────────────────────────
# 5,174 images — LARGE tier
if dataset_player is not None:
    print("=" * 60)
    print("TRAINING: player_isolator_v3 (5,174 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_player.location}/data.yaml",
        epochs=75,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="player_isolator_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        degrees=15,
        translate=0.15,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )
    metrics = YOLO("runs/detect/player_isolator_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"player_isolator_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: player_isolator_v3 — dataset not available")

In [ ]:
# ── Cell 3C — Train jersey_color_classifier_v3 ───────────────
# 1,232 images — MEDIUM tier
if dataset_color is not None:
    print("=" * 60)
    print("TRAINING: jersey_color_classifier_v3 (1,232 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_color.location}/data.yaml",
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="jersey_color_classifier_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    metrics = YOLO("runs/detect/jersey_color_classifier_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"jersey_color_classifier_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: jersey_color_classifier_v3 — dataset not available")

In [ ]:
# ── Cell 3D — Train number_region_detector_v3 ────────────────
# 556 images — MEDIUM tier
if dataset_numregion is not None:
    print("=" * 60)
    print("TRAINING: number_region_detector_v3 (556 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_numregion.location}/data.yaml",
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="number_region_detector_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    metrics = YOLO("runs/detect/number_region_detector_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"number_region_detector_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.5 else "FAIL — will be skipped in download")
else:
    print("SKIPPED: number_region_detector_v3 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CHUNK 3 DOWNLOAD — Run before Chunk 4
# ════════════════════════════════════════════════════════════════

download_if_pass("player_isolator_v3.pt", "player_isolator_v3")
download_if_pass("jersey_color_classifier_v3.pt", "jersey_color_classifier_v3")
download_if_pass("number_region_detector_v3.pt", "number_region_detector_v3")

print("\nCHUNK 3 COMPLETE — save files before Chunk 4")

# ================================================================
# CHUNK 4 — Augmentation Specialists (~60 min on A100)
# ================================================================
# Four models trained on the PRIMARY dataset with EXTREME augmentation
# settings to handle the hardest real-world conditions:
#   - motion_blur: fast camera pans, running players
#   - wide_angle: broadcast cameras, far-away shots (imgsz=1280)
#   - dark_jersey: navy/black jerseys, poor lighting
#   - partial_visibility: occluded players, cropped frames
#
# NOTE: No dedicated datasets exist for these conditions.
# Instead, we reuse the primary digit dataset and push augmentation
# to extreme values to simulate each condition.
# Download all files after this chunk — this is the FINAL chunk.

In [ ]:
# ── Cell 4A — Re-download primary dataset if needed ──────────
# Chunk 4 reuses the primary dataset from Chunk 1.
# If Colab disconnected between chunks, re-download it.

if 'dataset_primary' not in dir() or dataset_primary is None:
    print("Re-downloading primary dataset for specialist training...")
    dataset_primary = safe_download(
        "footballplayertracking", "jerseynumberdetectordigitdetector", 1
    )
else:
    print(f"Primary dataset already loaded: {dataset_primary.location}")

if dataset_primary is None:
    print("CRITICAL: Primary dataset unavailable — Chunk 4 cannot train.")
else:
    print("Primary dataset ready for specialist training")

In [ ]:
# ── Cell 4B — Train motion_blur_specialist_v3 ────────────────
# Extreme motion blur augmentation for fast camera pans + running players
# epochs=150, SMALL tier base + extreme motion augmentation
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: motion_blur_specialist_v3 (EXTREME blur augmentation)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=150,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="motion_blur_specialist_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.6,
        degrees=30,
        translate=0.2,
        scale=0.8,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.3,
        copy_paste=0.2,
        erasing=0.4,
        shear=5.0,
        perspective=0.001,
    )
    metrics = YOLO("runs/detect/motion_blur_specialist_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"motion_blur_specialist_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.4 else "FAIL — specialist threshold is 0.4")
else:
    print("SKIPPED: motion_blur_specialist_v3 — primary dataset not available")

In [ ]:
# ── Cell 4C — Train wide_angle_specialist_v3 ─────────────────
# Extreme scale augmentation for broadcast cameras + far-away shots
# imgsz=1280 (higher than standard 832 for tiny players in wide shots)
# epochs=150
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: wide_angle_specialist_v3 (imgsz=1280, EXTREME scale)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=150,
        imgsz=1280,
        batch=4,
        name="wide_angle_specialist_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.6,
        degrees=25,
        translate=0.2,
        scale=0.9,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.2,
        erasing=0.4,
        perspective=0.002,
    )
    metrics = YOLO("runs/detect/wide_angle_specialist_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"wide_angle_specialist_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.4 else "FAIL — specialist threshold is 0.4")
else:
    print("SKIPPED: wide_angle_specialist_v3 — primary dataset not available")

In [ ]:
# ── Cell 4D — Train dark_jersey_specialist_v3 ────────────────
# Focus on dark colored jerseys: navy, black, dark green
# Extreme HSV augmentation for poor lighting conditions
# epochs=150
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: dark_jersey_specialist_v3 (EXTREME HSV augmentation)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=150,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="dark_jersey_specialist_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.03,
        hsv_s=0.9,
        hsv_v=0.8,
        degrees=25,
        translate=0.2,
        scale=0.8,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.2,
        erasing=0.4,
    )
    metrics = YOLO("runs/detect/dark_jersey_specialist_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"dark_jersey_specialist_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.4 else "FAIL — specialist threshold is 0.4")
else:
    print("SKIPPED: dark_jersey_specialist_v3 — primary dataset not available")

In [ ]:
# ── Cell 4E — Train partial_visibility_specialist_v3 ─────────
# Extreme copy_paste + erasing for occluded/partially blocked jerseys
# epochs=150
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: partial_visibility_specialist_v3 (EXTREME erasing)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=150,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="partial_visibility_specialist_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.6,
        degrees=25,
        translate=0.2,
        scale=0.8,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.5,
        erasing=0.6,
    )
    metrics = YOLO("runs/detect/partial_visibility_specialist_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"partial_visibility_specialist_v3 mAP50: {map50:.3f}")
    print("PASS" if map50 >= 0.4 else "FAIL — specialist threshold is 0.4")
else:
    print("SKIPPED: partial_visibility_specialist_v3 — primary dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CHUNK 4 DOWNLOAD — Final chunk
# ════════════════════════════════════════════════════════════════
# Specialists use lower mAP50 threshold (0.4) since extreme
# augmentation intentionally hurts clean-set metrics.

download_if_pass("motion_blur_specialist_v3.pt", "motion_blur_specialist_v3", min_map50=0.4)
download_if_pass("wide_angle_specialist_v3.pt", "wide_angle_specialist_v3", min_map50=0.4)
download_if_pass("dark_jersey_specialist_v3.pt", "dark_jersey_specialist_v3", min_map50=0.4)
download_if_pass("partial_visibility_specialist_v3.pt", "partial_visibility_specialist_v3", min_map50=0.4)

print("\nCHUNK 4 COMPLETE — all specialist models downloaded")

# ================================================================
# FINAL SUMMARY
# ================================================================

In [ ]:
import os
from ultralytics import YOLO

print("=" * 60)
print("v3 OCR TRAINING COMPLETE — FINAL REPORT")
print("=" * 60)

# (model_filename, chunk_label, service_file, threshold)
all_v3_models = [
    ("jersey_ocr_v3_primary.pt",          "Chunk 1 — Multi-sport OCR",     "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.5),
    ("jersey_ocr_v3_secondary.pt",         "Chunk 1 — Multi-sport OCR",     "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.5),
    ("jersey_ocr_v3_ensemble.pt",          "Chunk 1 — Multi-sport OCR",     "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.5),
    ("basketball_ocr_v3.pt",               "Chunk 2 — Sport-specific OCR",  "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.5),
    ("football_ocr_v3.pt",                 "Chunk 2 — Sport-specific OCR",  "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.5),
    ("lacrosse_ocr_v3.pt",                 "Chunk 2 — Sport-specific OCR",  "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.5),
    ("player_isolator_v3.pt",              "Chunk 3 — Player isolation",    "roboflow_detector.py: detect_with_player_crops()", 0.5),
    ("jersey_color_classifier_v3.pt",      "Chunk 3 — Color classifier",   "roboflow_detector.py: detect_with_player_crops()", 0.5),
    ("number_region_detector_v3.pt",       "Chunk 3 — Number region",      "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.5),
    ("motion_blur_specialist_v3.pt",       "Chunk 4 — Motion blur",        "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.4),
    ("wide_angle_specialist_v3.pt",        "Chunk 4 — Wide angle",         "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.4),
    ("dark_jersey_specialist_v3.pt",       "Chunk 4 — Dark jersey",        "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.4),
    ("partial_visibility_specialist_v3.pt","Chunk 4 — Partial visibility", "roboflow_detector.py: _run_v3_ocr_on_crop()",  0.4),
]

passed = []
failed = []
for name, chunk, service, threshold in all_v3_models:
    if os.path.exists(name):
        size_mb = os.path.getsize(name) / 1024 / 1024
        try:
            m = YOLO(name)
            metrics = m.val()
            map50 = metrics.box.map50
            if map50 >= threshold:
                passed.append(f"  PASS: {name} ({size_mb:.1f}MB, mAP50: {map50:.3f}) — {chunk} — {service}")
            else:
                failed.append(f"  FAIL: {name} — mAP50: {map50:.3f} (below {threshold}) — {chunk}")
        except Exception:
            passed.append(f"  PASS: {name} ({size_mb:.1f}MB, mAP50: unknown) — {chunk} — {service}")
    else:
        run_path = f"runs/detect/{name.replace('.pt', '')}/weights/best.pt"
        if os.path.exists(run_path):
            try:
                m = YOLO(run_path)
                metrics = m.val()
                failed.append(f"  FAIL: {name} — mAP50: {metrics.box.map50:.3f} (not downloaded) — {chunk}")
            except Exception:
                failed.append(f"  FAIL: {name} — trained but validation failed — {chunk}")
        else:
            failed.append(f"  FAIL: {name} — MISSING (training failed or skipped) — {chunk}")

print(f"\nPASSED ({len(passed)}/13):")
for m in passed: print(m)

print(f"\nFAILED ({len(failed)}/13):")
for m in (failed or ["  (none)"]): print(m)

print()
print("GIT COMMANDS:")
print("cd playerJerseyIdentification-master")
print("git add app/model/*.pt")
print('git commit -m "Add v3 OCR models — Ali replacement"')
print("git push")
print()
print("After Railway deploys check:")
print("curl https://jersey-detection-production-d8d8.up.railway.app/health")
print("Look for roboflow_models_v3_ocr — all should show loaded")
print()
print("=" * 60)
print(f"v3 OCR PIPELINE: {len(passed)}/13 models ready")
print("=" * 60)